# GASKAP-OH

Notebook for accessing data via the Table Access Protocol (TAP) service attached to the database.

---

In [ ]:
import os
import requests
import pyvo as vo
from pyvo.auth import authsession, securitymethods
from astropy.io.votable import from_table, parse_single_table

In [ ]:
# Connect with TAP service

URL = "https://gaskap.aussrc.org/tap"
tap = vo.dal.TAPService(URL)

## Choose run

In [ ]:
# Fetch run

run_name = 'gaskap-oh-subregion2_sofiax_pipeline'
query = f"SELECT * FROM survey.run WHERE name='{run_name}'"
run_votable = tap.search(query)
run_table = run_votable.to_table()
run_table

## Query detections

In [ ]:
# Retrieve catalog as Astropy table

default_query = """
    SELECT d.*, ivo_string_agg(t.name || ': ' || t.description, '; ') AS tags FROM survey.detection d
    FULL JOIN survey.tag_detection td ON d.id = td.detection_id 
    LEFT JOIN survey.tag t ON t.id = td.tag_id
    GROUP BY d.id HAVING d.run_id = RUNID
"""
query = default_query.replace("RUNID", str(int(run_table['id'])))
query

In [ ]:
# Perform query

result = tap.search(query)
table = result.to_table()
table

## Download catalog

It is convenient to write the catalog (`astropy` Table object) to a number of file formats. Below we show how to export the table as a `.xml` votable file. For this, we convert the astropy table to a votable, but there are a number of write options for the astropy table:

https://docs.astropy.org/en/stable/io/ascii/write.html

<span style="font-weight: bold; color: #FF0000;">⚠ Update the `catalog_filename` value</span>

In [ ]:
# Filename for catalog

catalog_filename = "catalog.xml"

In [ ]:
# Download catalog table

votable = from_table(table)
votable.to_xml(catalog_filename)

## Download products

<span style="font-weight: bold; color: #FF0000;">⚠ Update the `products_filename` value</span>

We have written a function to download source products for an astropy table containing a list of detections. Here the arguments are:

* `table`: the astropy table of detections for which you would like to download product files
* `directory`: the write directory for the products
* `chunk_size`: size (bytes) of each chunk while streaming the download [default 8192 B]

In [ ]:
# Folder name for downloading data products (moment maps, spectra etc)

products_filename = 'products'

<span style="font-weight: bold; color: #FF0000;">⚠ Update the `username` and `password` values</span>

In [ ]:
# Credentials for the GASKAP-OH portal: https://gaskap.aussrc.org

username = 'gaskapsuper'
password = None

In [ ]:
# useful function for downloading table products (requires authentication)

def download_products(row, products_filename, chunk_size=8192):
    """Download products for a row of the table (a detection entry)
    
    """
    name = row['name']
    access_url = row['access_url']
    access_url = access_url.replace('survey.aussrc', 'gaskap.aussrc')
    votable = parse_single_table(access_url)
    product_table = votable.to_table()
    url = product_table[product_table['description'] == 'SoFiA-2 Detection Products'][0]['access_url']
    with requests.get(url, auth=(username, password), stream=True) as r:
        r.raise_for_status()
        with open(products_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
    print(f'Downloaded completed for {name}')
    return

def download_table_products(table, directory, chunk_size=8192):
    """Download WALLABY products from ADQL queried table

    """
    if not os.path.exists(directory):
        os.mkdir(directory)
    print(f'Saving products to {directory}')
    for row in table:
        name = row['source_name']
        products_filename = os.path.join(directory, f'{name}.tar')
        download_products(row, products_filename, chunk_size)
    print('Downloads complete')
    return

In [ ]:
# Write output products for a source

download_table_products(table[0:5], products_filename)

---